# Assignment 06: Clinic Data Wrangling Pipeline (Midterm)

Follow `README.md`: it names every file to save in `output/`, with its first line and line count. Replace each **TODO**, then restart the kernel and **Run All** before you commit, so the saved outputs show your work.

In [1]:
# Supplied setup: run this cell first and leave it unchanged.
from pathlib import Path
import hashlib
import json
import subprocess
import sys

import numpy as np
import pandas as pd

print("Python", sys.version.split()[0], "| NumPy", np.__version__, "| pandas", pd.__version__)


def find_assignment_dir(start):
    """The folder holding data_manifest.json: this one, a parent, or 06/mid-term below one."""
    for folder in (start, *start.parents):
        for candidate in (folder, folder / "06" / "mid-term"):
            if (candidate / "data_manifest.json").is_file():
                return candidate
    raise FileNotFoundError("Open this notebook from the assignment folder that holds data_manifest.json.")


BASE_DIR = find_assignment_dir(Path.cwd().resolve())
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)
PROFILE_PATH = OUTPUT_DIR / "table_profile.csv"
AUDIT_PATH = OUTPUT_DIR / "issue_audit.csv"
PATIENTS_SUMMARY_PATH = OUTPUT_DIR / "patients_summary.csv"
GENDER_CATEGORIES_PATH = OUTPUT_DIR / "gender_categories.csv"
VISITS_SUMMARY_PATH = OUTPUT_DIR / "visits_summary.csv"
DECISION_LOG_PATH = OUTPUT_DIR / "decision_log.csv"
A1C_SUMMARY_PATH = OUTPUT_DIR / "a1c_summary.csv"
MERGE_AUDIT_PATH = OUTPUT_DIR / "merge_audit.csv"
SBP_WIDE_SAMPLE_PATH = OUTPUT_DIR / "sbp_wide_sample.csv"
ANALYSIS_SAMPLE_PATH = OUTPUT_DIR / "analysis_sample.csv"

# The patients whose rows the two extracts hold (README Tasks 4.3 and 5.2).
SAMPLE_IDS = ["P00001", "P00004", "P00088", "P00260", "P00446", "P00487", "P00631", "P00851",
              "P02354", "P03836", "P05763", "P07510", "P14287", "P16486", "P17390", "P30001"]

# The data is generated, not committed: generate_data.py writes the same tables on every computer.
manifest = json.loads((BASE_DIR / "data_manifest.json").read_text(encoding="utf-8"))
if not all((DATA_DIR / name).is_file() for name in manifest["files"]):
    print(subprocess.run([sys.executable, "generate_data.py"], cwd=BASE_DIR, check=True,
                         capture_output=True, text=True).stdout.strip())


def read_raw(path):
    """A Parquet file with the dtypes it stores; a CSV file with every column as text, blanks kept as written."""
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path, dtype="string", keep_default_na=False)


raw = {Path(name).stem: read_raw(DATA_DIR / name) for name in manifest["files"]}
# The checksum covers each table's contents, written out as CSV text.
for name, expected in manifest["files"].items():
    text = raw[Path(name).stem].to_csv(index=False, lineterminator="\n")
    assert hashlib.sha256(text.encode("utf-8")).hexdigest() == expected["sha256"], (
        f"data/{name} differs from the generated file; delete the data folder and run this cell again.")
print(f"Verified {len(manifest['files'])} data files")
raw_snapshot = {name: table.copy(deep=True) for name, table in raw.items()}
list(raw)

Python 3.13.15 | NumPy 2.3.3 | pandas 3.0.5
Verified 5 data files


['patients', 'sites', 'visits_north', 'visits_south', 'a1c_wide']

## Cumulative midterm checkpoint (human review)

Lecture 01: Used shell command in task 1.1 to count date lines in sites and visits north/south .csv files. 

Lecture 02: Created a function 'profile' in task 1.2 to return a summary dictionary for each table. 

Lecture 03: The NumPy calculation after 4.3 gave per-visit means of 143.01 for 'baseline' and 137.15 for 'week12'. I choice 'axis=0' so that the per-visit means where calculated columnwise across patients rather than taking a mean of individual patient's visits. 

Lecture 04: Used '.reset_index(drop=True)' in task 3.1 when cleaning patient data to clean up row labels. 

Lecture 05: Used regex full string matching to validate visit dates in task 3.1 while cleaning patient data.

Lecture 06: Used melt in task 4.1 to change wide A1c records into long format. 

## Task 1: Inventory the files

### 1.1 Count lines from the terminal

Make `output/file_inventory.txt` in the terminal (README Task 1.1), then paste the command you ran into the `%%bash` cell below.

In [2]:
%%bash
wc -l data/*.csv > output/file_inventory.txt

### 1.2 Profile each table with a function

In [3]:
def profile(name, table):
    """Returns a summary dictionary for the given table."""
    return {
        "table": name,
        "rows": len(table),
        "columns": len(table.columns),
        "exact_duplicate_rows": table.duplicated().sum()
    }

profile_results = [profile(name, raw[Path(name).stem]) for name in manifest["files"]]
table_profile = pd.DataFrame(profile_results)
table_profile.to_csv(PROFILE_PATH, index=False)

## Task 2: Contract, stack, and audit

### 2.1 Write the data contract (human review)

patients

    Intended Grain: one row per unique patient
    Key: "patient_id" (single)
    Violations: exact duplicant patient rows that need to be dropped to avoid artifically inflating the data.

visits_north

    Intended Grain: one row per patient per visit
    Key: ["patient_id", "visit"] (composite)
    Violations: Dulpicate measurements and rechecks on the same visit label which need to be sorted and the last measurement kept. 

visits_south

    Intended Grain: one row per patient per visit
    Key: ["patient_id", "visit"] (composite)
    Violations: Dulpicate measurements and rechecks on the same visit label which need to be sorted and the last measurement kept. 

sites

    Intended Grain: one row per active clinic site
    Key: "site_code" (single)
    Violations: Retired site code versions that need to be filtered out.

a1c_wide

    Intended Grain: one row per patient in wide form with lab columns.
    Key: "patient_id"
    Violation: Need to alter formating for later merging.

Cardinality:

Visit to Patients: Many to One

    There will be multiple visits that need to be matched to each patient.

Patients to Site: Many to One

    There are many patients at each site.

Visits to A1c results: One to One

    Each visit is matched to one unique lab result. 
    


### 2.2 Stack the two visit files

In [4]:
visits_south = raw["visits_south"].copy(deep=True)
visits_south = visits_south.rename(columns={"wt_kg": "weight_kg"})
visits_south["source_file"] = "visits_south.csv"

visits_north = raw["visits_north"].copy(deep=True)
visits_north["source_file"] = "visits_north.csv"

visits_raw = pd.concat([visits_north, visits_south], ignore_index=True)

### 2.3 Audit the raw tables

In [5]:
patients_raw = raw["patients"]
sites_raw = raw["sites"]

valid_genders = {"F", "M", "X"}
stripped_gender = patients_raw["gender"].str.strip()
invalid_gender = (stripped_gender.isin(valid_genders))
empty_gender = (stripped_gender == "")
sentinel_year = (patients_raw["birth_year"] == "9999")
norm_site = patients_raw["site_code"].str.upper().str.replace("-", "", regex=False).str.replace(" ", "", regex=False)
invalid_site = (patients_raw["site_code"] != norm_site)

repeated_site = sites_raw.duplicated(subset="site_code", keep=False)

v_labels = visits_raw["visit"].str.strip().str.lower()
invalid_visit_lable = ~v_labels.isin(["baseline", "week12"])

sbp_sentinel = (visits_raw["sbp_mmhg"] == "999")
sbp_num = pd.to_numeric(visits_raw["sbp_mmhg"], errors="coerce")
sbp_out = (~sbp_sentinel) & (sbp_num.isna() | (sbp_num < 70) | (sbp_num > 250))

dates = visits_raw["visit_date"]
exact = dates.str.fullmatch(r'[0-9]{4}-[0-9]{2}-[0-9]{2}')
parsed = pd.to_datetime(dates.where(exact), errors="coerce")
date_failures = parsed.isna()

visit_map = {
    "bl": "baseline", "baseline": "baseline", "baseline ": "baseline",
    "week 12": "week12", "wk12": "week12", " week12": "week12",
    "week12": "week12", "WEEK12": "week12", "Wk12": "week12", "Baseline": "baseline"
}
mapped_visit = v_labels.map(visit_map)
repeated_visits = visits_raw.assign(_mapped_visit=mapped_visit).duplicated(subset=["patient_id", "_mapped_visit"], keep=False)


issue_audit = pd.DataFrame({
    "issue": [
        "patients: gender values that are not a code",
        "patients: empty gender",
        "patients: birth_year sentinel 9999",
        "patients: site codes needing normalization",
        "sites: rows witha repeated site_code",
        "visits: visit labels that are not baseline or week12",
        "visits: sbp sentinel 999",
        "visits: sbp outside 70 through 250",
        "visits: date parse failures",
        "visits: rows in repeated patient_id and visit pairs",
    ],
    "count": [
        invalid_gender.sum(),
        empty_gender.sum(),
        sentinel_year.sum(),
        invalid_site.sum(),
        repeated_site.sum(),
        invalid_visit_lable.sum(),
        sbp_sentinel.sum(),
        sbp_out.sum(),
        date_failures.sum(),
        repeated_visits.sum()
    ]
})

issue_audit.to_csv(AUDIT_PATH, index=False)

assert all(raw[name].equals(raw_snapshot[name]) for name in raw), "Auditing must not change raw."

## Task 3: Clean

### 3.1 Clean the patients

In [6]:
patients_clean = raw["patients"].copy(deep=True).drop_duplicates().reset_index(drop=True)
gender_dict = {
    "f": "F", "female": "F", "woman": "F",
    "m": "M", "male": "M", "man": "M",
    "x": "X", "nonbinary": "X", "non-binary": "X", "nb": "X"
}
patients_clean["gender"] = patients_clean["gender"].str.strip().str.lower().map(gender_dict).astype("category")
patients_clean["birth_year"] = pd.to_numeric(patients_clean["birth_year"].replace("9999", pd.NA), errors="coerce").astype("Int64")
patients_clean["site_code"] = patients_clean["site_code"].str.upper().str.replace("-", "", regex=False).str.replace(" ", "", regex=False)

patients_clean["gender"].cat.categories



Index(['F', 'M', 'X'], dtype='str')

In [7]:
patients_summary = pd.DataFrame({
    "metric": [
        "patients",
        "birth_year missing",
        "birth_year min",
        "birth_year max",
        "site_code values",
        "patients at N01",
        "patients at S02",
        "patients at E04"
    ],
    "value": [
        len(patients_clean),
        patients_clean["birth_year"].isna().sum(),
        patients_clean["birth_year"].min(),
        patients_clean["birth_year"].max(),
        patients_clean["site_code"].nunique(),
        (patients_clean["site_code"] == "N01").sum(),
        (patients_clean["site_code"] == "S02").sum(),
        (patients_clean["site_code"] == "E04").sum()
    ]
})

patients_summary.to_csv(PATIENTS_SUMMARY_PATH, index=False)

gender_counts = patients_clean["gender"].value_counts(dropna=False).reset_index()
gender_counts.columns = ["gender", "count"]
gender_counts["code"] = gender_counts["gender"].cat.codes
gender_counts.to_csv(GENDER_CATEGORIES_PATH, index=False)

### 3.2 Clean the visits

In [8]:
visits_clean = visits_raw.copy(deep=True).drop_duplicates().reset_index(drop=True)
print(visits_clean["visit"].value_counts(dropna=False))

visit_map = {
    "bl": "baseline", "baseline": "baseline", "baseline ": "baseline", "bl ": "baseline",
    "week 12": "week12", "wk12": "week12", " week12": "week12", "week12": "week12",
    "WEEK12": "week12", "Wk12": "week12", "Baseline": "baseline"
}
visits_clean["visit"] = visits_clean["visit"].str.strip().str.lower().map(visit_map)
print(visits_clean["visit"].value_counts(dropna=False))

vc_sbp_mmhg = pd.to_numeric(visits_clean["sbp_mmhg"], errors="coerce")
visits_clean["sbp_mmhg"] = vc_sbp_mmhg.where((vc_sbp_mmhg >= 70) & (vc_sbp_mmhg <= 250)).astype("Int64")

visits_clean["weight_kg"] = pd.to_numeric(visits_clean["weight_kg"], errors="coerce")

vc_dates = visits_clean["visit_date"]
vc_exact = vc_dates.str.fullmatch(r'[0-9]{4}-[0-9]{2}-[0-9]{2}')
vc_parsed = pd.to_datetime(vc_dates.where(vc_exact), format='%Y-%m-%d', errors='coerce')
visits_clean["visit_date"] = vc_parsed

visit
baseline     28496
week12       26498
BASELINE       255
Baseline       242
bl             235
baseline       228
BL             220
wk12           204
 week12        198
Week 12        194
WEEK12         186
Wk12           177
week 12        168
Name: count, dtype: int64[pyarrow]
visit
baseline    29676
week12      27625
Name: count, dtype: int64


In [9]:
rechecks = visits_clean[visits_clean.duplicated(subset=["patient_id", "visit"], keep=False)]
print(rechecks.sort_values(["patient_id", "visit", "visit_time"]))
visits_clean = visits_clean.sort_values("visit_time").drop_duplicates(subset=["patient_id", "visit"], keep="last")

      patient_id     visit visit_date visit_time  sbp_mmhg  weight_kg  \
26263     P00001  baseline 2026-03-06      14:15       141       52.4   
26262     P00001  baseline 2026-03-06      14:40       146       52.4   
4         P00004  baseline 2026-02-25      10:55       145      106.2   
5         P00004  baseline 2026-02-25      11:15       142      106.2   
84        P00088  baseline 2026-03-10      09:30       153       68.2   
...          ...       ...        ...        ...       ...        ...   
55438     P28232  baseline 2026-01-12      10:10       159       61.0   
56214     P28953  baseline 2026-01-21      10:30       153       74.8   
56213     P28953  baseline 2026-01-21      10:45       151       74.8   
25464     P29170  baseline 2026-03-02      11:20       130       78.0   
25465     P29170  baseline 2026-03-02      11:45       128       78.0   

            source_file  
26263  visits_south.csv  
26262  visits_south.csv  
4      visits_north.csv  
5      visits_north

In [10]:
visits_clean = visits_clean.sort_values(["patient_id", "visit"]).reset_index(drop=True)
visits_clean = visits_clean[["patient_id", "visit", "visit_date", "sbp_mmhg", "weight_kg", "source_file"]]
print(visits_clean["visit"].value_counts(dropna=False))

summary_data = [
    ("visits", len(visits_clean)),
    ("baseline visits", (visits_clean["visit"] == "baseline").sum()),
    ("week12 visits", (visits_clean["visit"] == "week12").sum()),
    ("visit_date missing", visits_clean["visit_date"].isna().sum()),
    ("sbp_mmhg missing", visits_clean["sbp_mmhg"].isna().sum()),
    ("weight_kg missing", visits_clean["weight_kg"].isna().sum()),
    ("visit_date min", visits_clean["visit_date"].min()),
    ("visit_date max", visits_clean["visit_date"].max()),
    ("sbp_mmhg min", visits_clean["sbp_mmhg"].min()),
    ("sbp_mmhg max", visits_clean["sbp_mmhg"].max()),
    ("weight_kg mean", round(visits_clean["weight_kg"].mean(), 2)),
    ("visits from visits_north.csv", (visits_clean["source_file"] == "visits_north.csv").sum()),
    ("visits from visits_south.csv", (visits_clean["source_file"] == "visits_south.csv").sum())
]
visits_summary = (pd.DataFrame(summary_data, columns=["metric", "value"]))

visits_summary.to_csv(VISITS_SUMMARY_PATH, index=False)

visit
baseline    29599
week12      27582
Name: count, dtype: int64


### 3.3 Record the decisions

In [11]:
issues_report = [
    (
        "patients",
        "exact duplicate rows",
        "remove duplicate rows while keeping the first occurrence.",
        "Removing dulpicates makes sure eah patient record is unique and prevents inflation of the data."
    ),
    (
        "patients",
        "gender spellings that are not a code",
        "Standardize gender spellings to a consistent code.",
        "Standardizing gender spellings ensures consistency and facilitates accurate analysis."
    ),
    (
        "patients",
        "site codes needing normalization",
        "Converted all site coes to Uppercase and removed any spaces or hyphens present.",
        "Standardizes the format of all clinic location codes across tables."
    ),
    (
        "patients",
        "birth_year sentinel 9999",
        "Replaced sentinel value 9999 with missing values and cast to Int64.",
        "Prevents unrealistic age values from altering analysis."
    ),
    (
        "vistis",
        "weight column named wt_kg in visits_south.csv",
        "Renamed the column to weight_kg during stack preparation.",
        "Ensures uniform column naming across different visit tables allowing for combination."
    ),
    (
        "visits",
        "sbp sentinel 999 and readings outside 70 through 250",
        "Converted sentinel value 999 to missing values and restricted readings to the valid range of 70 to 250.",
        "Ensures that only valid systolic blood pressure readings are used in analysis."
    ),
    (
        "visits",
        "impossible and loosely written visit dates",
        "Parsed date strings exactly matching YYYY-MM-DD format and validated against the calendar, making invalid entries missing.",
        "This removes impossible calendar dates to ensure accuracy.",
    ),
    (
        "visits",
        "repeated patient_id and visit pairs",
        "Sort by visit time keeping the last occurrence to remove duplicate patient_id and visit pairs and account for rechecks.",
        "Ensures that each patient-visit combination is unique and the most recent measurement is retained, preventing data inflation."
    ),
    (
        "visits",
        "repeated site_code rows",
        "Keep only the current site records and unique site codes.",
        "Ensures that each site code appears only once and reflects the current status."
    )
    
]

decision_table = pd.DataFrame(issues_report, columns=["table", "issue", "decision", "reason"])
decision_table

decision_table.to_csv(OUTPUT_DIR / "decision_log.csv", index=False)

## Task 4: Reshape and join

### 4.1 Melt the A1c results to long form

In [12]:
a1c_long = raw["a1c_wide"].melt(
    id_vars=["patient_id"],
    value_vars=["a1c_baseline", "a1c_week12"],
    var_name="visit",
    value_name="a1c_pct"
)

a1c_visit_map = {"a1c_baseline": "baseline", "a1c_week12": "week12"}
a1c_long["visit"] = a1c_long["visit"].map(a1c_visit_map)
print(a1c_long["a1c_pct"].dtype)
a1c_long = a1c_long.dropna(subset=["a1c_pct"]).reset_index(drop=True)


summary_rows = []
for visit in a1c_long["visit"].unique():
    count = (a1c_long["visit"] == visit).sum()
    mean_val = round(
        a1c_long.loc[a1c_long["visit"] == visit, "a1c_pct"].mean(), 2
    )
    summary_rows.append(
        {"visit": visit, "results": count, "a1c_pct_mean": mean_val}
    )
a1c_summary = pd.DataFrame(summary_rows)
print(a1c_summary)

a1c_summary.to_csv(A1C_SUMMARY_PATH, index=False)

float64
      visit  results  a1c_pct_mean
0  baseline    29107          8.15
1    week12    27653          7.63


### 4.2 Check the keys before joining

In [13]:
sites = raw["sites"]
try:
    patients_clean.merge(sites, on="site_code", how="left", validate="many_to_one")
except pd.errors.MergeError as err:
    print(f"Caught expected MergeError: {err}")

repeated_sites = sites[sites.duplicated(subset=["site_code"], keep=False)],
print(repeated_sites)

sites_current = sites[sites["record_status"] == "current"][["site_code", "site_name"]].reset_index(drop=True)
print(sites_current)

Caught expected MergeError: Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
 site_code
      N01
      S02 ...
(  site_code            site_name record_status
0       N01         North Clinic       current
1       N01  North Street Clinic       retired
2       S02     Southgate Clinic       retired
3       S02  South Valley Clinic       current,)
  site_code            site_name
0       N01         North Clinic
1       S02  South Valley Clinic
2       W03          West Clinic


In [14]:
v_a1c_merge = pd.merge(
    visits_clean, a1c_long, on=["patient_id", "visit"], how="outer", indicator=True
)
print(v_a1c_merge["_merge"].value_counts())

merge_audit = pd.DataFrame(
    {
        "check": [
            "visits with an A1c result",
            "visits without an A1c result",
            "A1c results without a visit",
            "visits whose patient is not in the registry",
            "registry patients with no visit",
            "registry patients whose site is not in the site list",
            "site list rows removed by the current-record rule",
        ],
        "count": [
            (v_a1c_merge["_merge"] == "both").sum(),
            (v_a1c_merge["_merge"] == "left_only").sum(),
            (v_a1c_merge["_merge"] == "right_only").sum(),
            (~visits_clean["patient_id"].isin(patients_clean["patient_id"])).sum(),
            (~patients_clean["patient_id"].isin(visits_clean["patient_id"])).sum(),
            (~patients_clean["site_code"].isin(sites["site_code"])).sum(),
            len(sites) - len(sites_current),
        ],
    }
)


merge_audit.to_csv(MERGE_AUDIT_PATH, index=False)

_merge
both          54006
left_only      3175
right_only     2754
Name: count, dtype: int64


### 4.3 Pivot blood pressure to one row per patient

In [15]:
sbp_wide = visits_clean.pivot(
    index="patient_id", columns="visit", values = "sbp_mmhg").reset_index()
sbp_wide.columns.name = None
sbp_wide["sbp_change"] = sbp_wide["week12"] - sbp_wide["baseline"]

sbp_wide_sample = sbp_wide[sbp_wide["patient_id"].isin(SAMPLE_IDS)].reset_index(drop=True)

sbp_wide_sample.to_csv(SBP_WIDE_SAMPLE_PATH, index=False)
sbp_wide_sample

,patient_id,baseline,week12,sbp_change
0,P00001,146,129,-17
1,P00004,142,<NA>,<NA>
2,P00088,152,147,-5
3,P00260,70,138,68
4,P00446,250,144,-106
5,P00487,<NA>,143,<NA>
6,P00851,<NA>,129,<NA>
7,P02354,127,125,-2
8,P03836,134,125,-9
9,P05763,161,151,-10


#### NumPy checkpoint (Lecture 03, human review)

From the full `sbp_wide`, select `baseline` and `week12` in that order, keeping only patients with both readings. Name this two-column table `complete_pairs`; `np.array(complete_pairs, dtype=float)` converts it to a NumPy array. Display its shape and calculate one NumPy mean per visit across patients. Save the outputs and explain your axis choice in the cumulative checkpoint's Lecture 03 bullet. No extra output file.

In [16]:
complete_pairs = sbp_wide[["baseline", "week12"]].dropna()
complete_array = np.array(complete_pairs, dtype=float)
display(complete_array)
print("Per-visit means (baseline, week12):", np.mean(complete_array, axis=0))

array([[146., 129.],
       [132., 129.],
       [148., 137.],
       ...,
       [135., 135.],
       [142., 138.],
       [144., 133.]], shape=(27149, 2))

Per-visit means (baseline, week12): [143.01337066 137.15061328]


### 4.4 Join the analysis table

In [17]:
analysis = visits_clean.merge(
    patients_clean, on="patient_id", how="left", validate="many_to_one"
)
analysis = analysis.merge(
    sites_current, on="site_code", how="left", validate="many_to_one"
)
analysis = analysis.merge(
    a1c_long, on = ["patient_id", "visit"], how="left", validate="one_to_one"
)

analysis = analysis.astype({
    "patient_id": "string",
    "visit": "string",
    "visit_date": "datetime64[us]",
    "sbp_mmhg": "Int64",
    "weight_kg": "float64",
    "a1c_pct": "float64",
    "gender": "string",
    "birth_year": "Int64",
    "site_code": "string",
    "site_name": "string",
    "source_file": "string",
})

analysis = analysis[
    [
        "patient_id",
        "visit",
        "visit_date",
        "sbp_mmhg",
        "weight_kg",
        "a1c_pct",
        "gender",
        "birth_year",
        "site_code",
        "site_name",
        "source_file",
    ]
]
analysis.dtypes

patient_id             string
visit                  string
visit_date     datetime64[us]
sbp_mmhg                Int64
weight_kg             float64
a1c_pct               float64
gender                 string
birth_year              Int64
site_code              string
site_name              string
source_file            string
dtype: object

## Task 5: Validate, save, and read back

### 5.1 Validate before saving

In [18]:
checks = pd.Series(
    {
        "anaysis row count matches visits_clean": len(analysis) == len(visits_clean),
        "no repeated patient_id and visit pairs": not analysis.duplicated(subset=["patient_id", "visit"]).any(),
        "all sbp between 70 and 250": analysis["sbp_mmhg"].dropna().between(70, 250).all(),
        "every site_name is current": analysis["site_name"].dropna().isin(sites_current["site_name"]).all(),
        "sbp_wide patient_id is unique": sbp_wide["patient_id"].is_unique,
        "raw tables equal its snapshots": all(raw[name].equals(raw_snapshot[name]) for name in raw
        )
    }
)

display(checks)
assert checks.all(), checks[~checks]

anaysis row count matches visits_clean    True
no repeated patient_id and visit pairs    True
all sbp between 70 and 250                True
every site_name is current                True
sbp_wide patient_id is unique             True
raw tables equal its snapshots            True
dtype: bool

### 5.2 Save the analysis extract

In [19]:
analysis_sample = analysis[analysis["patient_id"].isin(SAMPLE_IDS)].reset_index(drop=True)
analysis_sample.to_csv(ANALYSIS_SAMPLE_PATH, index=False)

visits_clean["patient_id"].isin(SAMPLE_IDS).sum() == len(analysis_sample)

np.True_

### 5.3 Read the file back

In [20]:
round_trip = pd.read_csv(
    ANALYSIS_SAMPLE_PATH,
    dtype={
        "patient_id": "string",
        "visit": "string",
        "sbp_mmhg": "Int64",
        "weight_kg": "float64",
        "a1c_pct": "float64",
        "gender": "string",
        "birth_year": "Int64",
        "site_code": "string",
        "site_name": "string",
        "source_file": "string",
    },
    parse_dates=["visit_date"],
)

print(round_trip.equals(analysis_sample))

True
